# Crash detector, walkthrough

Loads UMAFall, computes the ten window features, trains one fold of each model and plots what the detector sees. The reported numbers come from `train.py`; this notebook is for looking.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from data_umafall import load_all
from features import features, window_around_peak, resample, resultant, FEATURE_NAMES, rule_detector
from synthetic import dataset as synthetic_dataset, scenario

trials = [t for t in load_all(Path('data/umafall')) if 'phone' in t.signals]
len(trials), sum(t.label for t in trials)

## What a fall and a daily activity look like

In [ ]:
fall = next(t for t in trials if t.label == 1)
adl = next(t for t in trials if t.activity == 'Walking')
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for a, t in zip(ax, [fall, adl]):
    w = window_around_peak(t.signals['phone'])
    a.plot(w[:, 0], resultant(w), lw=1)
    a.axhline(6, ls='--', lw=0.8, color='orange')
    a.set_title(f'{t.activity} (label {t.label})')
    a.set_xlabel('s'); a.set_ylabel('g')
plt.tight_layout()

## Features

In [ ]:
X = np.array([features(window_around_peak(t.signals['phone'])) for t in trials])
y = np.array([t.label for t in trials])
df = pd.DataFrame(X, columns=FEATURE_NAMES); df['fall'] = y
df.groupby('fall').median().T

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for a, name in zip(axes.ravel(), FEATURE_NAMES):
    a.hist(df[df.fall == 0][name], bins=30, alpha=0.6, label='ADL')
    a.hist(df[df.fall == 1][name], bins=30, alpha=0.6, label='fall')
    a.set_title(name)
axes[0, 0].legend(); plt.tight_layout()

## The rule against the model, one leave-subjects-out fold

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
groups = np.array([t.subject for t in trials])
tr, te = next(GroupKFold(5).split(X, y, groups))
gbm = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.08, max_iter=200, class_weight='balanced', random_state=0).fit(X[tr], y[tr])
p = gbm.predict_proba(X[te])[:, 1]
rule = np.array([rule_detector(f)[0] for f in X[te]])
print('rule'); print(classification_report(y[te], rule, digits=3))
print('gbm'); print(classification_report(y[te], p >= 0.5, digits=3)); print('AUROC', round(roc_auc_score(y[te], p), 3))

## Synthetic rider scenarios

In [ ]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharey=True)
for a, name in zip(axes.ravel(), ['riding', 'braking', 'pothole', 'set_down', 'bag_drop', 'crash']):
    w = scenario(name, rng)
    a.plot(w[:, 0], resultant(w), lw=1)
    f = features(w)
    a.set_title(f"{name}  p={gbm.predict_proba(f[None])[0, 1]:.2f}")
plt.tight_layout()

## Which activities fool it

In [ ]:
acts = np.array([t.activity for t in trials])[te]
pred = p >= 0.5
pd.Series({a: pred[(acts == a) & (y[te] == 0)].mean() for a in sorted(set(acts[y[te] == 0]))}).sort_values(ascending=False).head(5)